# Marketing A/B Testing: Ad vs. PSA — A Rigorous Analysis

Goes beyond "p < 0.05" to include randomization/balance checks, prospective power analysis,
confidence intervals, and segmented/interaction effects.

**Dataset:** [Marketing A/B Testing](https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing) (Kaggle)


## 1. Business Objective

**Question:** Should we show commercial ads instead of PSAs (public service announcements) to the full user base?

**Primary metric:** Conversion rate (CR)

**Minimum effect worth acting on (set *before* looking at results):** an increase of at least **0.5 percentage
points** in CR. This threshold anchors the later interpretation — statistical significance alone isn't the bar;
the effect also has to clear this practical threshold to justify the change.

**Guardrail metric:** none available in this dataset (no cost/spend data) — noted as a limitation.


## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (
    proportion_effectsize, proportions_ztest, confint_proportions_2indep
)
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.anova import anova_lm
import statsmodels.formula.api as smf

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
np.random.seed(42)

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']


## 3. Load & Clean Data

Update `DATA_PATH` below to wherever you saved the CSV after downloading it from Kaggle.

In [ ]:
DATA_PATH = 'data/marketing_AB.csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# Drop the junk index column some Kaggle exports include
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Rename to clean, code-friendly names
df = df.rename(columns={
    'user id': 'user_id',
    'test group': 'test_group',
    'total ads': 'total_ads',
    'most ads day': 'most_ads_day',
    'most ads hour': 'most_ads_hour'
})

# Fix dtypes
df['test_group'] = df['test_group'].astype('category')
df['converted'] = df['converted'].astype(int)
df['most_ads_day'] = df['most_ads_day'].astype('category')

df.info()


## 4. Data Quality Checks

Basic sanity checks before trusting anything downstream.

In [ ]:
print("Full-row duplicates:", df.duplicated().sum())
print("Duplicate user_ids:", df.duplicated('user_id').sum())
print("\nNulls per column:\n", df.isna().sum())
print("\nUnique values in 'converted':", df['converted'].unique())


In [ ]:
print("Group sizes and conversion rates:")
df.groupby('test_group')['converted'].agg(['mean', 'count'])


## 5. Randomization / Balance Check

Before trusting a "the ad group converted better" result, check whether the two groups are actually comparable —
i.e., whether which group a user landed in is related to *when* they were exposed to ads. If it is, that's a
red flag for the randomization.

**Important interpretation note:** with ~588K rows, even tiny, practically meaningless differences will often
come back as "statistically significant" on a chi-square test. So we look at both the p-value *and* the actual
proportions — not the p-value alone.

In [ ]:
contingency_day = pd.crosstab(df['test_group'], df['most_ads_day'])
chi2_day, p_day, _, _ = chi2_contingency(contingency_day)

contingency_hour = pd.crosstab(df['test_group'], df['most_ads_hour'])
chi2_hour, p_hour, _, _ = chi2_contingency(contingency_hour)

print(f"Day-of-week association p-value: {p_day:.6g}")
print(f"Hour-of-day association p-value: {p_hour:.6g}")


In [ ]:
# Look at the actual shape of the distributions, not just the p-value
day_props = pd.crosstab(df['test_group'], df['most_ads_day'], normalize='index').reindex(columns=weekday_order)
day_props.T.plot(kind='bar', figsize=(9,4))
plt.ylabel('Share of group')
plt.title('Day-of-week distribution by group (normalized)')
plt.tight_layout()
plt.show()

day_props


**How to read this:** if the two rows have a similar overall *shape* (peaks/dips on the same days), the
groups are practically comparable even if the chi-square p-value is tiny — the association is a large-sample
statistical artifact, not a real confound. If the shapes look meaningfully different, that's worth investigating
further before trusting the main test.

## 6. Power Analysis / Sample Size

Two questions:
1. Given our stated MDE (0.5pp) and baseline PSA conversion rate, what sample size per group would we need
   for 80% power at α=0.05?
2. Does our actual sample size clear that bar?

In [ ]:
cr_psa = df.loc[df['test_group'] == 'psa', 'converted'].mean()
cr_mde = cr_psa + 0.005   # our 0.5pp business threshold

alpha = 0.05
power_target = 0.80

effect_size = proportion_effectsize(cr_mde, cr_psa)

analysis = NormalIndPower()
required_n = analysis.solve_power(
    effect_size=effect_size, power=power_target, alpha=alpha, alternative='larger', ratio=1
)

n_psa_actual = (df['test_group'] == 'psa').sum()
n_ad_actual = (df['test_group'] == 'ad').sum()

print(f"Baseline (psa) conversion rate: {cr_psa:.4%}")
print(f"Required sample size per group (80% power, MDE=0.5pp): {int(np.ceil(required_n)):,}")
print(f"Actual psa group size: {n_psa_actual:,}")
print(f"Actual ad group size:  {n_ad_actual:,}")

if n_psa_actual >= required_n:
    print("-> Adequately powered to detect our stated business threshold.")
else:
    print("-> UNDERPOWERED relative to our stated business threshold.")


## 7. Exploratory Data Analysis

In [ ]:
cr_day_sample = (
    df.groupby(['most_ads_day', 'test_group'], observed=True)['converted']
      .mean()
      .unstack()
      .reindex(weekday_order)
)
cr_day_sample.plot(kind='bar', figsize=(9,4))
plt.ylabel('Conversion rate')
plt.title('Conversion rate by weekday and group')
plt.tight_layout()
plt.show()

cr_day_sample


In [ ]:
cr_hour_ad = df.loc[df['test_group'] == 'ad'].groupby('most_ads_hour')['converted'].mean()
cr_hour_ad.plot(kind='bar', figsize=(9,4), color='#DD8452')
plt.ylabel('Conversion rate')
plt.title('Conversion rate by hour (ad group)')
plt.tight_layout()
plt.show()


## 8. Main Hypothesis Test

**H0:** CR(ad) = CR(psa)
**H1:** CR(ad) > CR(psa)  (one-sided, matching our directional business question)

**Method:** two-proportion z-test — the correct test for a binary outcome (not a t-test, which is for
continuous means).

In [ ]:
conv = df.groupby('test_group', observed=True)['converted'].sum()
n_obs = df.groupby('test_group', observed=True)['converted'].count()

z_stat, p_val = proportions_ztest(conv, n_obs, alternative='larger')

# Two-sided 95% CI on the difference (kept two-sided by convention, even though the test is one-sided —
# a more purist match would use a one-sided 95%/two-sided 90% interval; noting the choice here explicitly)
ci_low, ci_high = confint_proportions_2indep(conv['ad'], n_obs['ad'], conv['psa'], n_obs['psa'])

rate_ad = conv['ad'] / n_obs['ad']
rate_psa = conv['psa'] / n_obs['psa']
abs_lift = (rate_ad - rate_psa) * 100
rel_lift = (rate_ad - rate_psa) / rate_psa

main_result = pd.DataFrame({
    'z_stat': [z_stat],
    'p_value': [p_val],
    'abs_lift_pp': [abs_lift],
    'rel_lift_pct': [rel_lift * 100],
    'ci_lower_pp': [ci_low * 100],
    'ci_upper_pp': [ci_high * 100],
})
main_result


In [ ]:
print(f"Does the CI clear our 0.5pp business threshold?")
print("YES" if ci_low * 100 > 0.5 else "NO — interpret with caution")


## 9. Robustness Check: Bootstrap

A non-parametric cross-check on the analytic z-test/CI above.

In [ ]:
ad_vals = df.loc[df['test_group'] == 'ad', 'converted'].values
psa_vals = df.loc[df['test_group'] == 'psa', 'converted'].values

n_boot = 2000
boot_diffs = np.empty(n_boot)

for i in range(n_boot):
    ad_sample = np.random.choice(ad_vals, size=len(ad_vals), replace=True)
    psa_sample = np.random.choice(psa_vals, size=len(psa_vals), replace=True)
    boot_diffs[i] = ad_sample.mean() - psa_sample.mean()

boot_ci_low, boot_ci_high = np.percentile(boot_diffs, [2.5, 97.5])
boot_p = np.mean(boot_diffs <= 0)

print(f"Bootstrap 95% CI on absolute difference: [{boot_ci_low*100:.4f} pp, {boot_ci_high*100:.4f} pp]")
print(f"Analytic 95% CI was:                      [{ci_low*100:.4f} pp, {ci_high*100:.4f} pp]")
print(f"Share of bootstrap draws with diff <= 0 (bootstrap p-value proxy): {boot_p:.4f}")


## 10. Segmentation

Two segmentation angles:
1. **Day of week** — does the effect hold on every day, or concentrate on specific days?
2. **Engagement tier** (proxy for user tenure) — this dataset has no native "new vs. returning" flag, so we
   proxy it using `total_ads` seen, bucketed into tertiles. This is an explicit, disclosed modeling choice,
   not a ground-truth label.

For each, we test per-segment effects **and** run a formal interaction test — testing whether the effect
*differs* by segment, rather than just comparing separate within-segment p-values (which is a common and
misleading shortcut).

In [ ]:
df['engagement_tier'] = pd.qcut(df['total_ads'], q=3, labels=['low', 'medium', 'high'])
df['engagement_tier'].value_counts()


### 10a. By day of week

In [ ]:
pvals, ci_lower, ci_upper, lifts = [], [], [], []

for day in weekday_order:
    sub = df[df['most_ads_day'] == day]
    counts = sub.groupby('test_group', observed=True)['converted'].sum()
    nobs = sub.groupby('test_group', observed=True)['converted'].count()

    pvals.append(proportions_ztest(counts, nobs)[1])
    low, high = confint_proportions_2indep(counts['ad'], nobs['ad'], counts['psa'], nobs['psa'])
    ci_lower.append(low * 100)
    ci_upper.append(high * 100)
    lifts.append((counts['ad']/nobs['ad'] - counts['psa']/nobs['psa']) * 100)

reject, pvals_fdr, _, _ = multipletests(pvals, method='fdr_bh')

daily_results = pd.DataFrame({
    'day': weekday_order,
    'p_value': pvals,
    'p_value_fdr': pvals_fdr,
    'lift_pp': lifts,
    'ci_lower_pp': ci_lower,
    'ci_upper_pp': ci_upper,
    'significant_after_correction': reject
})
daily_results


### 10b. By engagement tier (tenure proxy)

In [ ]:
pvals_e, ci_lower_e, ci_upper_e, lifts_e = [], [], [], []

for tier in ['low', 'medium', 'high']:
    sub = df[df['engagement_tier'] == tier]
    counts = sub.groupby('test_group', observed=True)['converted'].sum()
    nobs = sub.groupby('test_group', observed=True)['converted'].count()

    pvals_e.append(proportions_ztest(counts, nobs)[1])
    low, high = confint_proportions_2indep(counts['ad'], nobs['ad'], counts['psa'], nobs['psa'])
    ci_lower_e.append(low * 100)
    ci_upper_e.append(high * 100)
    lifts_e.append((counts['ad']/nobs['ad'] - counts['psa']/nobs['psa']) * 100)

reject_e, pvals_fdr_e, _, _ = multipletests(pvals_e, method='fdr_bh')

engagement_results = pd.DataFrame({
    'tier': ['low', 'medium', 'high'],
    'p_value': pvals_e,
    'p_value_fdr': pvals_fdr_e,
    'lift_pp': lifts_e,
    'ci_lower_pp': ci_lower_e,
    'ci_upper_pp': ci_upper_e,
    'significant_after_correction': reject_e
})
engagement_results


### 10c. Formal interaction tests

Rather than relying on the tables above alone, we test explicitly whether the treatment effect *differs* by
segment — using both a linear probability model (OLS/ANOVA, matching common practice) and a logistic model
with an interaction term (the more textbook-correct approach for a binary outcome), as a cross-check against
each other.

In [ ]:
# Linear probability model / ANOVA — day interaction
model_ols_day = smf.ols('converted ~ C(test_group) * C(most_ads_day)', data=df).fit()
anova_lm(model_ols_day)


In [ ]:
# Logistic regression with interaction — day, as a cross-check on the OLS/ANOVA result above
model_logit_day = smf.logit('converted ~ C(test_group) * C(most_ads_day)', data=df).fit(disp=False)

interaction_terms = [t for t in model_logit_day.params.index if ':' in t]
pd.DataFrame({
    'coef': model_logit_day.params[interaction_terms],
    'p_value': model_logit_day.pvalues[interaction_terms]
})


In [ ]:
# Logistic regression with interaction — engagement tier
model_logit_eng = smf.logit('converted ~ C(test_group) * C(engagement_tier)', data=df).fit(disp=False)

interaction_terms_eng = [t for t in model_logit_eng.params.index if ':' in t]
pd.DataFrame({
    'coef': model_logit_eng.params[interaction_terms_eng],
    'p_value': model_logit_eng.pvalues[interaction_terms_eng]
})


**Interpretation:** a significant interaction term is the statistically correct evidence that the treatment
effect genuinely varies by segment — not just "significant in one segment, not in another" (which is a common
and misleading way to claim a segment difference without actually testing for one).

## 11. Conclusions

*(Fill in after running against your actual downloaded data — the structure below is ready to go.)*

- **Randomization/balance check:** [did the day/hour distributions look comparable across groups?]
- **Power analysis:** [was the sample size sufficient to detect our 0.5pp threshold?]
- **Main result:** [absolute lift, relative lift, 95% CI — does the CI clear 0.5pp?]
- **Bootstrap:** [did it agree with the analytic CI?]
- **Segmentation:** [did the interaction tests show a genuine difference in effect by day / engagement tier,
  or was the effect roughly uniform?]
- **Recommendation:** ship / don't ship / gather more data — tied explicitly to whether the CI clears the
  pre-registered 0.5pp threshold, not just statistical significance.

### Limitations
- No native "new vs. returning" user flag — engagement tier was proxied via `total_ads` exposure tertiles.
- No guardrail/cost metric available in this dataset.
- Group sizes are heavily imbalanced (~96% ad / ~4% psa), which reduces power for segment-level splits in
  particular — smaller segments should be interpreted with that in mind.
